# 📌 Engineering Notes: Python ↔ SQL Server & Data Exports
### 1. Connecting Python to SQL Server
Libraries: Use sqlalchemy + pyodbc + urllib.parse to encode connection strings safely.

Instance Formatting: For named instances like SQLEXPRESS, use raw strings (r".\SQLEXPRESS") or double backslashes to avoid escape errors.

Modern Security Flags: Always pair Encrypt=yes; with TrustServerCertificate=yes; when working locally to satisfy mandatory encryption without certificate authority failures.

Connection String Pattern:

                import urllib
                from sqlalchemy import create_engine
                
                conn_str = (
                    "DRIVER={ODBC Driver 17 for SQL Server};"
                    r"SERVER=.\SQLEXPRESS;"
                    "DATABASE=nyc_taxi_data;"
                    "Trusted_Connection=yes;"
                    "Encrypt=yes;"
                    "TrustServerCertificate=yes;"
                )
                engine = create_engine(
                    f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}"
                )

---

### 2. Executing Queries, Summaries & CSV Exports
* **Direct DataFrame Ingestion:** Use `pd.read_sql_query(text(sql), conn)` within a managed context (`with engine.connect() as conn:`).
* **Exporting Cleanly:** Save directly via `df.to_csv("exports/summary.csv", index=False)` to produce version-controllable tabular files for GitHub.
* **Large Datasets:** Use `chunksize` in `pd.read_sql_query` to stream millions of rows without memory exhaustion.

---

### 3. Key Points Solved Along the Way
* **Windows Paths in Python:** Never end a string with a bare backslash (`"path\"`); Python treats `\"` as an escaped quotation mark, triggering an unterminated string syntax error. Use forward slashes (`/`) or `pathlib.Path`.
* **SQL Server Datatypes:** `DATEPART(HOUR, ...)` fails on `DATE` types (`Msg 9810`). Columns storing timestamps must be defined or cast as `DATETIME` or `DATETIME2(0)`.
* **Bulk Ingestion Alignment:** `BULK INSERT` fails or inserts 0 rows if table columns and CSV fields do not match in both count and ordinal position. Intercepting the load with a staging `VIEW` isolates pipeline audit columns (`source_file`, `ingested_at`).

---

<FollowUp label="Want to package these export functions into an automated CLI pipeline script?" query="Show me how to package this SQL-to-CSV logic into a reusable Python script that loops over all our analytical queries and saves individual reports automatically."/>

In [8]:
#!pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [15]:
import urllib
import pandas as pd
from sqlalchemy import create_engine, text

In [17]:
# Exact server instance from SSMS
SERVER = r".\SQLEXPRESS"
DATABASE = "nyc_taxi_data"

In [19]:
# Notice the addition of TrustServerCertificate=yes and Encrypt=yes
CONNECTION_STRING = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

In [21]:
# URL-encode the connection string for SQLAlchemy
params = urllib.parse.quote_plus(CONNECTION_STRING)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


In [23]:
# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION AS version;")).fetchone()
    print("Connection successful!")
    print(result[0])

Connection successful!
Microsoft SQL Server 2022 (RTM) - 16.0.1000.6 (X64) 
	Oct  8 2022 05:58:25 
	Copyright (C) 2022 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Home Single Language 10.0 <X64> (Build 26200: ) (Hypervisor)



In [41]:


# --- 2. SQL Query ---
# Example: Aggregating hourly rush-hour metrics from the bronze table
sql_query = """
SELECT 
    DATEPART(HOUR, CAST(tpep_pickup_datetime AS DATETIME2)) AS pickup_hour,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
    ROUND(AVG(fare_amount), 2) AS avg_fare_usd,
    ROUND(SUM(total_amount), 2) AS total_revenue_usd
FROM bronze.yellow_tripdata
WHERE fare_amount > 0 
  AND trip_distance > 0
GROUP BY DATEPART(HOUR, CAST(tpep_pickup_datetime AS DATETIME2))
ORDER BY pickup_hour ASC;
"""

In [43]:
with engine.connect() as connection:
    # Read directly into a Pandas DataFrame
    df = pd.read_sql_query(sql=text(sql_query), con=connection)
df.head()

,pickup_hour,total_trips,avg_distance_miles,avg_fare_usd,total_revenue_usd
0,0,4977,2.89,12.26,75020.6


In [45]:
# --- 3. Execute & Export to CSV ---
output_dir = "C:/Users/User/nyc-taxi-data-engineering-project/data/exports"
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(output_dir, "hourly_trip_summary.csv")

# Write to CSV
df.to_csv(output_csv_path, index=False, encoding="utf-8")

print(f"Export successful: {len(df)} rows written to {output_csv_path}")

Export successful: 1 rows written to C:/Users/User/nyc-taxi-data-engineering-project/data/exports\hourly_trip_summary.csv
